========================================================================================
Get the residual only from test set and save them for ML 
========================================================================================
Save the residuals separately for training and test set for PCA training and test
========================================================================================

In [ ]:
import sys
sys.path.append('working_path')

import numpy as np
import pandas as pd
import os

from sklearn.model_selection import KFold, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.multioutput import MultiOutputClassifier, MultiOutputRegressor
from sklearn.compose import ColumnTransformer

In [ ]:
df = pd.read_csv('working_path/cross_reversed.csv')

In [ ]:
"""
Define variables:
X set is sleep health
y set is depression
Age and sex are confounds 
"""
# Define variables
Age = ['age']
Sex = ['Sex']
sleep = ['sleep_variables']
N_12 = ['n_12_depression_variables']
RDS_4 = ['rds4_depression_variables']
depression = N_12 + RDS_4

X_set = depression
y_set = sleep
confounds = Age # Sex not included for normalization 

# Extract numerical data for X_set, y_set, and confounds 
X_data = df[X_set].values
y_data = df[y_set].values

In [5]:
# Define feature group for scaling
features_to_scale = RDS_4 + confounds
features_to_skip = N_12 + Sex

# Setup the preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), features_to_scale)
    ],
    remainder='passthrough' # N_12 and sex are binary
)

In [ ]:
"""
To keep the same structure of the data in sleep-related depression, the dataset will be splitted to 5 folds, each of them will be applied for multivariate GLM and the other 4 for ML.
Residuals from multivaraite GLM will be the target for later ML.
"""
# Define cross-validation strageties
# Define the CV for the entire dataset
n_splits_entire = 5

random_seed = 42

Outer_cv_entire = KFold(n_splits=n_splits_entire, shuffle=True, random_state=random_seed)

# Define grid of potential regularization parameters
alpha_grid = {'estimator__alpha': np.logspace(-4, 0, 10)}

In [ ]:
'''
The residuals will be calculated and saved separately for training and test set for later PCA
'''
# Train multivariateGLM in inner CV and get the residuals for the test sets in outer CV
for i, (ML_idx, GLM_idx) in enumerate(Outer_cv_entire.split(df)):
    
    # Split data
    df_GLM_train = df.iloc[GLM_idx]
    df_ML_test = df.iloc[ML_idx]

    # Select variables and Scale
    X_train_raw = df_GLM_train[features_to_scale + features_to_skip]
    X_test_raw = df_ML_test[features_to_scale + features_to_skip]

    X_train_s = preprocessor.fit_transform(X_train_raw)
    X_test_s = preprocessor.transform(X_test_raw) 

    # Define target 
    Y_train = y_data[GLM_idx]
    Y_test = y_data[ML_idx]

    # Train the Ridge Regressor 
    ridge = MultiOutputRegressor(Ridge())
    grid_ord = GridSearchCV(ridge, param_grid=alpha_grid, scoring='neg_mean_squared_error')
    grid_ord.fit(X_train_s, Y_train)

    # Calculate residuals on the traning set
    Y_train_pred = grid_ord.predict(X_train_s)
    res_train = Y_train - Y_train_pred

    # Project onto the test set
    Y_test_pred = grid_ord.predict(X_test_s)
    
    # Calculate Residuals for test set
    res_test = Y_test - Y_test_pred

    # Print out summary
    print(f'Projected residuals for SH for fold {i+1}: {res_test})')

    # Assign variable names for residuals
    res_cols = [f"{col}_res" for col in sleep]

    # Assign the residuals to the main df for the matching rows
    df_GLM = df.iloc[GLM_idx].copy()
    df_ML = df.iloc[ML_idx].copy()

    df_GLM[res_cols] = res_train
    df_ML[res_cols] = res_test

    # Save to csv
    filename_GLM = f'save_path/SH_residuals_training_fold_{i+1}.csv'
    filename_ML = f'save_path/SH_residuals_test_fold_{i+1}.csv'
    df_GLM.to_csv(filename_GLM, index=False)
    df_ML.to_csv(filename_ML, index=False)